In [14]:
import os, warnings, time
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
DATA_DIR = "."          # folder holding the four Kaggle CSV files
OUT_DIR = "outputs"     # tables + figures are written here
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 160, "display.max_columns", 30)
WEATHER = ["AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]
PAPER = {"RMSE": 0.46, "MAE": 0.229, "R2": 0.99}   # tuned X-LSTM-EO, paper abstract / results
PAPER_BASE = {"RMSE": 2.2, "MAE": 1.15, "R2": 0.67}  # baseline LSTM
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.2.2 | numpy 1.26.4


In [15]:
# 1. Data audit and the paper's merge error
FMT = {1: "%d-%m-%Y %H:%M", 2: "%Y-%m-%d %H:%M:%S"}   # generation-file date formats

def load_raw(plant):
    g = pd.read_csv(f"{DATA_DIR}/Plant_{plant}_Generation_Data.csv")
    w = pd.read_csv(f"{DATA_DIR}/Plant_{plant}_Weather_Sensor_Data.csv")
    return g, w

def correct_merge(plant):
    # Explicit date formats per file; join on plant + timestamp (weather has ONE sensor key).
    g, w = load_raw(plant)
    g["DATE_TIME"] = pd.to_datetime(g["DATE_TIME"], format=FMT[plant])
    w["DATE_TIME"] = pd.to_datetime(w["DATE_TIME"], format="%Y-%m-%d %H:%M:%S")
    m = g.merge(w.drop(columns="SOURCE_KEY"), on=["DATE_TIME", "PLANT_ID"], how="inner")
    return m.sort_values(["DATE_TIME", "SOURCE_KEY"]).reset_index(drop=True), g, w

def paper_merge_plant1():
    # Re-creates the paper's table: generation dates read MONTH-first (the error).
    g, w = load_raw(1)
    g["DATE_TIME"] = pd.to_datetime(g["DATE_TIME"], format="mixed", dayfirst=False)
    w["DATE_TIME"] = pd.to_datetime(w["DATE_TIME"])
    m = g.merge(w.drop(columns="SOURCE_KEY"), on=["DATE_TIME", "PLANT_ID"])
    return m.sort_values(["DATE_TIME", "SOURCE_KEY"]).reset_index(drop=True)

paper_df = paper_merge_plant1()
p1, g1, w1 = correct_merge(1)
p2, g2, w2 = correct_merge(2)

audit = pd.DataFrame({
    "rows": [len(paper_df), len(p1), len(p2)],
    "days": [paper_df.DATE_TIME.dt.date.nunique(), p1.DATE_TIME.dt.date.nunique(), p2.DATE_TIME.dt.date.nunique()],
    "inverters": [paper_df.SOURCE_KEY.nunique(), p1.SOURCE_KEY.nunique(), p2.SOURCE_KEY.nunique()],
    "mean DC_POWER": [paper_df.DC_POWER.mean(), p1.DC_POWER.mean(), p2.DC_POWER.mean()],
}, index=["Plant 1 – paper's (flawed) merge", "Plant 1 – correct merge", "Plant 2 – correct merge"]).round(2)
display(audit)

# --- sanity assertions tying this notebook to Phase 1 -------------------------
assert len(paper_df) == 45680 and paper_df.DATE_TIME.dt.date.nunique() == 23   # paper's table
assert len(p1) == 68774 and p1.DATE_TIME.dt.date.nunique() == 34               # correct table
assert abs(paper_df.DC_POWER.mean() - 3197.18) < 0.05                           # paper's Table 4 mean
print("OK – paper's 45,680-row table and Table-4 mean (3,197.18) reproduced; correct merge keeps all 34 days.")

# DC/AC ratio and other data-quality checks (Phase 1, Section 2.3)
for name, d in [("Plant 1", p1), ("Plant 2", p2)]:
    day = d[d.AC_POWER > 0]
    print(f"{name}: DC/AC ratio = {day.DC_POWER.sum()/day.AC_POWER.sum():.2f} | duplicates = {d.duplicated().sum()} "
          f"| NaNs = {int(d.isna().sum().sum())} | zero-DC rows with irradiation>0.1 = {int(((d.DC_POWER==0)&(d.IRRADIATION>0.1)).sum())}")

,rows,days,inverters,mean DC_POWER
Plant 1 – paper's (flawed) merge,45680,23,22,3197.18
Plant 1 – correct merge,68774,34,22,3147.18
Plant 2 – correct merge,67698,34,22,246.70


OK – paper's 45,680-row table and Table-4 mean (3,197.18) reproduced; correct merge keeps all 34 days.
Plant 1: DC/AC ratio = 10.23 | duplicates = 0 | NaNs = 0 | zero-DC rows with irradiation>0.1 = 63
Plant 2: DC/AC ratio = 1.02 | duplicates = 0 | NaNs = 0 | zero-DC rows with irradiation>0.1 = 3774


In [16]:
# Figure: days kept / lost by the paper's merge
fig, ax = plt.subplots(figsize=(11, 3.2))
full = p1.groupby(p1.DATE_TIME.dt.date).size()
kept = paper_df.groupby(paper_df.DATE_TIME.dt.date).size().reindex(full.index, fill_value=0)
ax.bar(range(len(full)), full.values, color="#d1d5db", label="correct merge")
ax.bar(range(len(full)), kept.values, color="#2563eb", label="paper's merge (month-first dates)")
ax.set_xticks(range(0, len(full), 3)); ax.set_xticklabels([str(d)[5:] for d in full.index[::3]], rotation=45)
ax.set_ylabel("rows per day"); ax.set_title("Plant 1: days retained by the paper's merge (23/34)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig1_days_lost.png"); plt.show()

In [17]:
# 2. Evaluation helpers
# Metrics: RMSE, MAE, R² (the three the paper reports that are well-defined; #COV and EC are defined only in the paper, so I do not reproduce them – see #Phase 1).
# Every metric is reported on 'all rows' (to compare with the paper) #and on 'daytime rows' (irradiation > 0), where the prediction problem is #non-trivial.
# Target unit: `DC_POWER / 1000`, the only scaling that makes the #paper's RMSE 0.46 plausible.


def metrics(y, p, day_mask=None):
    out = {"RMSE": mean_squared_error(y, p) ** .5, "MAE": mean_absolute_error(y, p), "R2": r2_score(y, p)}
    if day_mask is not None and day_mask.sum() > 1:
        out.update({"RMSE_day": mean_squared_error(y[day_mask], p[day_mask]) ** .5,
                    "MAE_day": mean_absolute_error(y[day_mask], p[day_mask]),
                    "R2_day": r2_score(y[day_mask], p[day_mask])})
    return out

def make_models(seed=SEED):
    # Untuned / default-style baselines. Scaling is inside the pipeline => fitted on training data only.
    return {
        "Linear (irradiation only)": make_pipeline(LinearRegression()),
        "Linear (3 weather inputs)": make_pipeline(StandardScaler(), LinearRegression()),
        "Decision tree": DecisionTreeRegressor(random_state=seed),
        "Random forest": RandomForestRegressor(n_estimators=150, min_samples_leaf=5, n_jobs=-1, random_state=seed),
        "Gradient boosting (HistGB)": HistGradientBoostingRegressor(random_state=seed),
        "MLP (32-16)": make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32, 16), early_stopping=True, max_iter=400, random_state=seed)),
    }

def cols_for(name, cols):
    return ["IRRADIATION"] if name == "Linear (irradiation only)" else cols

def chrono_split(dates, n_train=24, n_val=5):
    # Whole days, in time order: first n_train days -> train, next n_val -> validation, rest -> test.
    days = np.array(sorted(pd.unique(pd.Series(dates).dt.normalize())))
    tr, va, te = days[:n_train], days[n_train:n_train + n_val], days[n_train + n_val:]
    d = pd.Series(dates).dt.normalize().values
    return np.isin(d, tr), np.isin(d, va), np.isin(d, te), (tr, va, te)

def tune(factory, grid, Xtr, ytr, Xva, yva):
    # Pick hyper-parameters on the VALIDATION days only (paper tuned on training MSE - the flaw).
    best, best_rmse, rows = None, np.inf, []
    for params in ParameterGrid(grid):
        m = factory(**params).fit(Xtr, ytr)
        r = mean_squared_error(yva, m.predict(Xva)) ** .5
        rows.append({**params, "val_RMSE": r})
        if r < best_rmse: best, best_rmse = params, r
    return best, pd.DataFrame(rows).sort_values("val_RMSE")

In [18]:
# 3. Experiment A – replicate the paper's (apparent) protocol
# Paper-style table (45,680 rows, 23 days), target `DC_POWER/1000`, 'random 80/20 split', three weather inputs, 'untuned' sklearn models, 5 different split seeds (mean ± sd).
# This tests whether the paper's accuracy needs an LSTM.


Xp = paper_df[WEATHER].values; yp = paper_df["DC_POWER"].values / 1000
dayp = (paper_df["IRRADIATION"].values > 0)
ts = paper_df["DATE_TIME"]

# Irreducible floor for ANY weather-only model: spread of the 22 inverters around their per-timestamp mean
grp_mean = pd.Series(yp).groupby(ts.values).transform("mean").values
floor = {"RMSE": mean_squared_error(yp, grp_mean) ** .5, "MAE": mean_absolute_error(yp, grp_mean), "R2": r2_score(yp, grp_mean)}
print("Floor (in-sample per-timestamp mean):", {k: round(v, 3) for k, v in floor.items()},
      f"| distinct input vectors: {len(np.unique(Xp, axis=0))} of {len(Xp)} rows")

rng_seeds = [0, 1, 2, 3, 4]
rows = []
for name in make_models():
    for s in rng_seeds:
        idx = np.random.RandomState(s).permutation(len(yp)); cut = int(.8 * len(yp))
        tr, te = idx[:cut], idx[cut:]
        m = make_models(s)[name]; c = cols_for(name, WEATHER); ci = [WEATHER.index(x) for x in c]
        m.fit(Xp[tr][:, ci], yp[tr]); r = metrics(yp[te], m.predict(Xp[te][:, ci]), dayp[te])
        rows.append({"model": name, "seed": s, **r})
A = pd.DataFrame(rows)
A_tab = A.groupby("model")[["RMSE", "MAE", "R2", "RMSE_day"]].agg(["mean", "std"]).round(3)
A_tab = A_tab.loc[list(make_models())]
display(A_tab)
A_tab.to_csv(f"{OUT_DIR}/expA_random_split_paper_table.csv")
best_A = A.groupby("model").RMSE.mean().min()
print(f"Best untuned baseline RMSE = {best_A:.3f}  vs paper X-LSTM-EO = {PAPER['RMSE']}  vs paper baseline LSTM = {PAPER_BASE['RMSE']}")
print(f"Floor RMSE = {floor['RMSE']:.3f}: the paper's 0.46 is only {100*(PAPER['RMSE']/floor['RMSE']-1):.1f}% above the best any weather-only model could ever do.")

Floor (in-sample per-timestamp mean): {'RMSE': 0.425, 'MAE': 0.159, 'R2': 0.989} | distinct input vectors: 2102 of 45680 rows


RMSE           MAE            R2        RMSE_day       
                             mean    std   mean    std   mean    std     mean    std
model                                                                               
Linear (irradiation only)   0.552  0.023  0.267  0.004  0.982  0.001    0.733  0.030
Linear (3 weather inputs)   0.549  0.023  0.260  0.004  0.982  0.001    0.730  0.030
Decision tree               0.457  0.025  0.170  0.005  0.987  0.001    0.609  0.033
Random forest               0.457  0.025  0.170  0.005  0.987  0.001    0.609  0.033
Gradient boosting (HistGB)  0.466  0.024  0.183  0.004  0.987  0.001    0.620  0.032
MLP (32-16)                 0.512  0.024  0.214  0.006  0.984  0.001    0.682  0.032

Best untuned baseline RMSE = 0.457  vs paper X-LSTM-EO = 0.46  vs paper baseline LSTM = 2.2
Floor RMSE = 0.425: the paper's 0.46 is only 8.1% above the best any weather-only model could ever do.


In [19]:
# 4. Experiment B – corrected protocol (per-inverter, same task as the paper)
# Correct merge (all 34 days, 68,774 rows), **chronological split by whole days** (days 1–24 train · 25–29 validation · 30–34 test, test never touched until the end), hyper-parameters chosen on 'validation RMSE', final models refit on train+validation and evaluated once on the test days.
# Inputs are still the three weather variables, so the numbers are directly comparable in kind with the paper's task.
# Also shown: the *random* 80/20 split on the same correct table to isolate the effect of the split.


d1 = p1.copy(); y1 = d1.DC_POWER.values / 1000; X1 = d1[WEATHER].values; day1 = d1.IRRADIATION.values > 0
tr, va, te, (dtr, dva, dte) = chrono_split(d1.DATE_TIME)
print(f"train days {len(dtr)} ({pd.Timestamp(dtr[0]).date()}..{pd.Timestamp(dtr[-1]).date()}) | val {len(dva)} | test {len(dte)} "
      f"({pd.Timestamp(dte[0]).date()}..{pd.Timestamp(dte[-1]).date()})  rows: {tr.sum()}/{va.sum()}/{te.sum()}")

GRIDS = {
    "Decision tree": (lambda **p: DecisionTreeRegressor(random_state=SEED, **p), {"max_depth": [4, 6, 8, 12, None], "min_samples_leaf": [5, 20, 50]}),
    "Random forest": (lambda **p: RandomForestRegressor(n_estimators=150, n_jobs=-1, random_state=SEED, **p), {"min_samples_leaf": [5, 20, 50], "max_features": [1.0, 0.67]}),
    "Gradient boosting (HistGB)": (lambda **p: HistGradientBoostingRegressor(random_state=SEED, **p), {"learning_rate": [0.03, 0.1], "max_leaf_nodes": [8, 16, 31], "l2_regularization": [0, 1.0]}),
    "MLP (32-16)": (lambda **p: make_pipeline(StandardScaler(), MLPRegressor(early_stopping=True, max_iter=400, random_state=SEED, **p)),
                    {"hidden_layer_sizes": [(16,), (32, 16), (64, 32)], "alpha": [1e-4, 1e-2]}),
}

def run_chrono(X, y, day, tr, va, te, cols_idx_fn, label):
    rows, best_params, preds = [], {}, {}
    for name in make_models():
        ci = cols_idx_fn(name)
        Xtr, Xva, Xte = X[tr][:, ci], X[va][:, ci], X[te][:, ci]
        if name in GRIDS:
            fac, grid = GRIDS[name]; bp, _ = tune(fac, grid, Xtr, y[tr], Xva, y[va]); m = fac(**bp)
        else:
            bp, m = {}, make_models()[name]
        m.fit(np.vstack([Xtr, Xva]), np.r_[y[tr], y[va]])             # refit on train+val
        p = m.predict(Xte); preds[name] = p; best_params[name] = bp
        rows.append({"model": name, **metrics(y[te], p, day[te])})
    return pd.DataFrame(rows).set_index("model"), best_params, preds

ci_w = lambda name: [2] if name == "Linear (irradiation only)" else [0, 1, 2]
B_chrono, B_params, B_preds = run_chrono(X1, y1, day1, tr, va, te, ci_w, "chrono")

# random 80/20 on the SAME correct table (isolates the split effect)
idx = np.random.RandomState(SEED).permutation(len(y1)); cut = int(.8 * len(y1)); rtr, rte = idx[:cut], idx[cut:]
rows = []
for name in make_models():
    ci = ci_w(name); fac_params = B_params.get(name, {})
    m = GRIDS[name][0](**fac_params) if name in GRIDS else make_models()[name]
    m.fit(X1[rtr][:, ci], y1[rtr]); rows.append({"model": name, **metrics(y1[rte], m.predict(X1[rte][:, ci]), day1[rte])})
B_random = pd.DataFrame(rows).set_index("model")

# per-timestamp floor on test days (oracle): predict the mean of the inverters at that timestamp
tt = d1.DATE_TIME.values[te]; yt = y1[te]
floor_te = pd.Series(yt).groupby(tt).transform("mean").values
floor_row = metrics(yt, floor_te, day1[te])

B = pd.concat({"Chronological (test days 30-34)": B_chrono, "Random 80/20 (same data)": B_random}, axis=1).round(3)
display(B[[("Chronological (test days 30-34)", c) for c in ["RMSE", "MAE", "R2", "RMSE_day"]] +
          [("Random 80/20 (same data)", c) for c in ["RMSE", "MAE", "R2", "RMSE_day"]]])
print("Oracle floor on test days (22-inverter spread):", {k: round(v, 3) for k, v in floor_row.items()})
print("Selected hyper-parameters (validation RMSE):"); [print("  ", k, v) for k, v in B_params.items() if v]
B.to_csv(f"{OUT_DIR}/expB_per_inverter_chrono_vs_random.csv")
gap = (B_chrono.RMSE / B_random.RMSE - 1) * 100
print("RMSE inflation, chronological vs random split (%):", gap.round(1).to_dict())

train days 24 (2020-05-15..2020-06-07) | val 5 | test 5 (2020-06-13..2020-06-17)  rows: 47698/10560/10516


Chronological (test days 30-34)                        Random 80/20 (same data)                       
                                                      RMSE    MAE     R2 RMSE_day                     RMSE    MAE     R2 RMSE_day
model                                                                                                                            
Linear (irradiation only)                            0.635  0.275  0.973    0.853                    0.606  0.271  0.978    0.806
Linear (3 weather inputs)                            0.636  0.273  0.973    0.856                    0.604  0.266  0.978    0.804
Decision tree                                        0.610  0.209  0.975    0.822                    0.559  0.208  0.981    0.746
Random forest                                        0.596  0.203  0.976    0.804                    0.531  0.188  0.983    0.709
Gradient boosting (HistGB)                           0.588  0.194  0.977    0.793                    0.550  0.206  0.982    0.734
MLP (32-16)                                          0.591  0.198  0.977    0.796                    0.569  0.223  0.980    0.760

Oracle floor on test days (22-inverter spread): {'RMSE': 0.54, 'MAE': 0.153, 'R2': 0.98, 'RMSE_day': 0.729, 'MAE_day': 0.278, 'R2_day': 0.963}
Selected hyper-parameters (validation RMSE):
   Decision tree {'max_depth': 6, 'min_samples_leaf': 50}
   Random forest {'max_features': 0.67, 'min_samples_leaf': 50}
   Gradient boosting (HistGB) {'l2_regularization': 1.0, 'learning_rate': 0.1, 'max_leaf_nodes': 8}
   MLP (32-16) {'alpha': 0.0001, 'hidden_layer_sizes': (16,)}
RMSE inflation, chronological vs random split (%): {'Linear (irradiation only)': 4.8, 'Linear (3 weather inputs)': 5.3, 'Decision tree': 9.2, 'Random forest': 12.3, 'Gradient boosting (HistGB)': 7.1, 'MLP (32-16)': 3.8}


In [20]:
# Figure: test-day predictions for one inverter-averaged view + residual by hour
test_df = d1[te].copy(); test_df["y"] = yt
for n in ["Linear (3 weather inputs)", "Random forest", "Gradient boosting (HistGB)"]: test_df[n] = B_preds[n]
plant_ts = test_df.groupby("DATE_TIME")[["y", "Linear (3 weather inputs)", "Random forest", "Gradient boosting (HistGB)"]].mean()
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
sub = plant_ts.iloc[:96 * 3]
ax[0].plot(sub.index, sub.y, "k", lw=2, label="measured (mean of 22 inverters)")
for n, c in zip(["Linear (3 weather inputs)", "Random forest", "Gradient boosting (HistGB)"], ["#ef4444", "#2563eb", "#16a34a"]): ax[0].plot(sub.index, sub[n], c, lw=1, label=n)
ax[0].set_title("First 3 test days – corrected protocol"); ax[0].set_ylabel("DC_POWER / 1000"); ax[0].legend(fontsize=7); ax[0].tick_params(axis="x", rotation=30)
hr = test_df.DATE_TIME.dt.hour
for n, c in zip(["Linear (3 weather inputs)", "Random forest", "Gradient boosting (HistGB)"], ["#ef4444", "#2563eb", "#16a34a"]):
    ax[1].plot(range(24), (test_df[n] - test_df.y).groupby(hr).apply(lambda s: np.sqrt((s ** 2).mean())).reindex(range(24)).values, c, label=n)
ax[1].set_title("RMSE by hour of day (test days)"); ax[1].set_xlabel("hour"); ax[1].set_ylabel("RMSE"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig2_test_predictions.png"); plt.show()

In [21]:
# 5. Experiment C – a genuine forecasting task (plant level)
#Phase 1 showed the paper's task is a same-time 'estimation' and that 22 inverters share each weather vector. Here I fix both:

# One target per input: the plant-level target is the mean DC power of the reporting inverters at each timestamp.
# A real horizon 'h' (15 min and 1 h ahead). Features use only information available at time *t*: weather at *t*, power at *t*, time-of-day (sin/cos), and irradiation at *t−1*. No future weather is used (that would require a numerical weather forecast).
# Missing 15-min slots are not interpolated; rows whose inputs or target slot are missing are dropped (counts are printed).
# Reference models: 'persistence' (ŷ = power at *t*) and 'climatology' (mean of the training power at the same time of day) – the standard yardsticks a forecast must beat. Skill = 1 − RMSE/RMSE_persistence (daytime rows).
# `h = 0` is the paper's nowcast (weather → power, same time) for comparison.

def plant_frame(merged, weather_df):
    # One row per 15-min slot on the FULL grid; power = mean DC over reporting inverters (/1000).
    pw = merged.groupby("DATE_TIME").agg(power=("DC_POWER", lambda s: s.mean() / 1000), n_inv=("DC_POWER", "size"))
    wx = weather_df.set_index("DATE_TIME")[WEATHER]
    grid = pd.date_range(merged.DATE_TIME.min().normalize(), merged.DATE_TIME.max().normalize() + pd.Timedelta("23h45min"), freq="15min")
    df = pd.DataFrame(index=grid).join(pw).join(wx)
    return df

def make_supervised(df, h):
    f = df.copy()
    f["tod_sin"] = np.sin(2 * np.pi * (f.index.hour * 60 + f.index.minute) / 1440)
    f["tod_cos"] = np.cos(2 * np.pi * (f.index.hour * 60 + f.index.minute) / 1440)
    f["irr_lag1"] = f["IRRADIATION"].shift(1)
    f["target"] = f["power"].shift(-h)
    f["irr_target"] = f["IRRADIATION"].shift(-h)          # used ONLY to define the daytime mask
    f["power_t"] = f["power"]
    feats = WEATHER + ["tod_sin", "tod_cos", "irr_lag1"] + (["power_t"] if h > 0 else [])
    need = feats + ["target", "irr_target"]
    n0 = len(f); f = f.dropna(subset=need)
    return f, feats, n0 - len(f)

class Climatology:
    # Mean training power at the same slot of day (target slot), a standard no-skill reference.
    def fit(self, tod_idx, y): self.m = pd.Series(y).groupby(tod_idx).mean(); return self
    def predict(self, tod_idx): return pd.Series(tod_idx).map(self.m).fillna(0).values

def run_forecast(df, h, label, seed=SEED, tune_models=True):
    f, feats, dropped = make_supervised(df, h)
    tr_, va_, te_, (a, b, c) = chrono_split(pd.Series(f.index), 24, 5)
    X, y = f[feats].values, f["target"].values; day = (f["irr_target"].values > 0)
    res, preds, bps = {}, {}, {}
    tgt_slot = ((f.index + h * pd.Timedelta("15min")).hour * 60 + (f.index + h * pd.Timedelta("15min")).minute).values
    if h > 0:
        preds["Persistence"] = f["power_t"].values[te_]
    clim = Climatology().fit(tgt_slot[tr_ | va_], y[tr_ | va_]); preds["Climatology"] = clim.predict(tgt_slot[te_])
    cols_all = list(range(len(feats)))
    cand = {"Ridge (all features)": lambda **p: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13)))}
    cand.update({k: GRIDS[k][0] for k in ["Random forest", "Gradient boosting (HistGB)", "MLP (32-16)"]})
    for name, fac in cand.items():
        grid = GRIDS.get(name, (None, {}))[1] if tune_models else {}
        bp = tune(fac, grid, X[tr_], y[tr_], X[va_], y[va_])[0] if grid else {}
        m = fac(**bp).fit(X[tr_ | va_], y[tr_ | va_]); preds[name] = m.predict(X[te_]); bps[name] = bp
    out = pd.DataFrame({n: metrics(y[te_], p, day[te_]) for n, p in preds.items()}).T
    ref = "Persistence" if h > 0 else "Climatology"
    out["skill_vs_" + ref.lower() + "_day"] = 1 - out["RMSE_day"] / out.loc[ref, "RMSE_day"]
    info = dict(h=h, rows=len(f), dropped=dropped, train_days=len(a), val_days=len(b), test_days=len(c), test_rows=int(te_.sum()))
    return out, info, preds, (f, te_, y, day, bps)

plant1_ts = plant_frame(p1, w1); plant2_ts = plant_frame(p2, w2)
print("slots on grid:", len(plant1_ts), "| slots with power:", int(plant1_ts.power.notna().sum()), "| with weather:", int(plant1_ts.IRRADIATION.notna().sum()))

C_res = {}
for h in [0, 1, 4]:
    t0 = time.time()
    out, info, preds, ctx = run_forecast(plant1_ts, h, f"P1 h={h}")
    C_res[h] = (out, info, preds, ctx)
    print(f"\n=== Plant 1, horizon h={h} step(s) = {15*h} min | {info} | {time.time()-t0:.0f}s")
    display(out.round(3))
    out.round(4).to_csv(f"{OUT_DIR}/expC_plant1_h{h}.csv")

slots on grid: 3264 | slots with power: 3157 | with weather: 3182

=== Plant 1, horizon h=0 step(s) = 0 min | {'h': 0, 'rows': 3148, 'dropped': 116, 'train_days': 24, 'val_days': 5, 'test_days': 5, 'test_rows': 478} | 9s


,RMSE,MAE,R2,RMSE_day,MAE_day,R2_day,skill_vs_climatology_day
Climatology,1.369,0.705,0.871,1.846,1.282,0.751,0.000
Ridge (all features),0.300,0.194,0.994,0.396,0.286,0.989,0.785
Random forest,0.230,0.108,0.996,0.311,0.195,0.993,0.832
Gradient boosting (HistGB),0.236,0.114,0.996,0.318,0.206,0.993,0.827
MLP (32-16),0.245,0.130,0.996,0.329,0.214,0.992,0.822



=== Plant 1, horizon h=1 step(s) = 15 min | {'h': 1, 'rows': 3137, 'dropped': 127, 'train_days': 24, 'val_days': 5, 'test_days': 5, 'test_rows': 476} | 14s


,RMSE,MAE,R2,RMSE_day,MAE_day,R2_day,skill_vs_persistence_day
Persistence,1.083,0.565,0.920,1.459,1.026,0.844,0.000
Climatology,1.371,0.707,0.871,1.849,1.285,0.749,-0.267
Ridge (all features),1.043,0.630,0.925,1.388,0.973,0.859,0.049
Random forest,0.982,0.488,0.934,1.324,0.887,0.871,0.093
Gradient boosting (HistGB),0.988,0.566,0.933,1.323,0.889,0.872,0.093
MLP (32-16),0.976,0.504,0.935,1.314,0.874,0.873,0.100



=== Plant 1, horizon h=4 step(s) = 60 min | {'h': 4, 'rows': 3116, 'dropped': 148, 'train_days': 24, 'val_days': 5, 'test_days': 5, 'test_rows': 472} | 14s


,RMSE,MAE,R2,RMSE_day,MAE_day,R2_day,skill_vs_persistence_day
Persistence,1.908,1.154,0.752,2.564,2.071,0.520,0.000
Climatology,1.378,0.713,0.871,1.853,1.289,0.750,0.277
Ridge (all features),1.556,1.126,0.835,2.009,1.603,0.706,0.217
Random forest,1.278,0.665,0.889,1.719,1.200,0.785,0.330
Gradient boosting (HistGB),1.272,0.743,0.890,1.705,1.210,0.788,0.335
MLP (32-16),1.252,0.684,0.893,1.682,1.203,0.794,0.344


In [22]:
# 5.1 Uncertainty: block bootstrap over test days
#The test set is only 5 days, so a single point estimate is fragile. I resample 'whole test days with replacement' (2,000 draws) and report 95 % intervals for the daytime RMSE and for the skill vs. persistence (h = 1).
#This is deliberately conservative: 5 blocks give wide intervals.

def day_bootstrap(f, te_, y, preds, names, ref, n=2000, seed=SEED):
    idx = f.index[te_]; days = idx.normalize().values; ud = np.unique(days)
    dm = (f["irr_target"].values[te_] > 0); yt = y[te_]
    rng = np.random.RandomState(seed); res = {k: [] for k in names}; skill = {k: [] for k in names if k != ref}
    for _ in range(n):
        pick = rng.choice(ud, len(ud)); sel = np.concatenate([np.where((days == d) & dm)[0] for d in pick])
        r = {k: np.sqrt(np.mean((yt[sel] - preds[k][sel]) ** 2)) for k in names}
        for k in names: res[k].append(r[k])
        for k in skill: skill[k].append(1 - r[k] / r[ref])
    q = lambda a: np.percentile(a, [2.5, 97.5])
    return ({k: q(v) for k, v in res.items()}, {k: q(v) for k, v in skill.items()})

out1, info1, preds1, (f1, te1, y_1, day_1, bps1) = C_res[1]
ci_rmse, ci_skill = day_bootstrap(f1, te1, y_1, preds1, list(preds1), "Persistence")
bt = pd.DataFrame({"RMSE_day": out1.RMSE_day, "CI_low": {k: v[0] for k, v in ci_rmse.items()}, "CI_high": {k: v[1] for k, v in ci_rmse.items()},
                   "skill_low": {k: v[0] for k, v in ci_skill.items()}, "skill_high": {k: v[1] for k, v in ci_skill.items()}}).round(3)
display(bt); bt.to_csv(f"{OUT_DIR}/expC_bootstrap_h1.csv")

,RMSE_day,CI_low,CI_high,skill_low,skill_high
Climatology,1.849,1.668,2.010,-0.325,-0.222
Gradient boosting (HistGB),1.323,1.191,1.438,0.067,0.110
MLP (32-16),1.314,1.166,1.440,0.080,0.122
Persistence,1.459,1.294,1.596,NaN,NaN
Random forest,1.324,1.155,1.472,0.077,0.110
Ridge (all features),1.388,1.257,1.506,0.027,0.066


In [23]:
## 6. Experiment D – external check on Plant 2 and stress tests
# Plant 2 (different plant, different date format, DC/AC ≈ 1.02) is run through the *identical* pipeline: if the conclusions are artefacts of Plant 1 they should not transfer.

D_res = {}
for h in [0, 1]:
    out, info, preds, ctx = run_forecast(plant2_ts, h, f"P2 h={h}")
    D_res[h] = out; print(f"\n=== Plant 2, h={h} | {info}"); display(out.round(3)); out.round(4).to_csv(f"{OUT_DIR}/expD_plant2_h{h}.csv")


=== Plant 2, h=0 | {'h': 0, 'rows': 3253, 'dropped': 11, 'train_days': 24, 'val_days': 5, 'test_days': 5, 'test_rows': 480}


,RMSE,MAE,R2,RMSE_day,MAE_day,R2_day,skill_vs_climatology_day
Climatology,0.129,0.066,0.759,0.171,0.116,0.526,0.000
Ridge (all features),0.100,0.067,0.855,0.130,0.096,0.727,0.241
Random forest,0.085,0.033,0.895,0.113,0.058,0.793,0.340
Gradient boosting (HistGB),0.087,0.033,0.891,0.115,0.059,0.785,0.327
MLP (32-16),0.089,0.040,0.886,0.117,0.065,0.776,0.313



=== Plant 2, h=1 | {'h': 1, 'rows': 3247, 'dropped': 17, 'train_days': 24, 'val_days': 5, 'test_days': 5, 'test_rows': 479}


,RMSE,MAE,R2,RMSE_day,MAE_day,R2_day,skill_vs_persistence_day
Persistence,0.110,0.058,0.824,0.146,0.101,0.653,0.000
Climatology,0.129,0.066,0.758,0.171,0.116,0.525,-0.170
Ridge (all features),0.102,0.061,0.849,0.134,0.095,0.708,0.083
Random forest,0.098,0.049,0.861,0.130,0.086,0.726,0.112
Gradient boosting (HistGB),0.098,0.055,0.861,0.129,0.087,0.728,0.115
MLP (32-16),0.098,0.054,0.860,0.130,0.090,0.726,0.112


In [24]:
# ---- Stress tests on the Plant-1 nowcast (h=0): input noise and irrelevant extra dimensions ----
out0, info0, preds0, (f0, te0, y0, day0, bps0) = C_res[0]
feats0 = WEATHER + ["tod_sin", "tod_cos", "irr_lag1"]
Xs, ys = f0[feats0].values, f0["target"].values
trm = chrono_split(pd.Series(f0.index), 24, 5)
trv, tem = (trm[0] | trm[1]), trm[2]
fac = {"Ridge": lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13))),
       "Random forest": lambda: GRIDS["Random forest"][0](**bps0["Random forest"]),
       "HistGB": lambda: GRIDS["Gradient boosting (HistGB)"][0](**bps0["Gradient boosting (HistGB)"]),
       "MLP": lambda: GRIDS["MLP (32-16)"][0](**bps0["MLP (32-16)"])}
dm0 = day0[tem]

# (a) Gaussian noise on the TEST weather inputs, sigma = fraction of each feature's training std
rng = np.random.RandomState(SEED); sig = [0, .05, .1, .2, .4]; fitted = {k: v().fit(Xs[trv], ys[trv]) for k, v in fac.items()}
sd = Xs[trv].std(0); rows = []
for s in sig:
    Xn = Xs[tem].copy(); Xn[:, :3] += rng.normal(0, 1, Xn[:, :3].shape) * sd[:3] * s
    for k, m in fitted.items(): rows.append({"noise_sigma": s, "model": k, "RMSE_day": mean_squared_error(ys[tem][dm0], m.predict(Xn)[dm0]) ** .5})
noise = pd.DataFrame(rows).pivot(index="noise_sigma", columns="model", values="RMSE_day")

# (b) add k irrelevant N(0,1) features to BOTH train and test (dimensionality stress)
rows = []
for k_extra in [0, 10, 50, 200]:
    R = np.random.RandomState(SEED); Ztr = R.normal(size=(trv.sum(), k_extra)); Zte = R.normal(size=(tem.sum(), k_extra))
    for k, v in fac.items():
        m = v().fit(np.hstack([Xs[trv], Ztr]), ys[trv]); p = m.predict(np.hstack([Xs[tem], Zte]))
        rows.append({"extra_noise_features": k_extra, "model": k, "RMSE_day": mean_squared_error(ys[tem][dm0], p[dm0]) ** .5})
dim = pd.DataFrame(rows).pivot(index="extra_noise_features", columns="model", values="RMSE_day")

display(noise.round(3)); display(dim.round(3))
noise.round(4).to_csv(f"{OUT_DIR}/expD_noise.csv"); dim.round(4).to_csv(f"{OUT_DIR}/expD_dimensionality.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
noise.plot(ax=ax[0], marker="o"); ax[0].set_title("Test-input noise (daytime RMSE)"); ax[0].set_xlabel("noise σ (× feature std)")
dim.plot(ax=ax[1], marker="o"); ax[1].set_title("Added irrelevant features (daytime RMSE)"); ax[1].set_xlabel("# extra noise features")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig3_stress.png"); plt.show()

model,HistGB,MLP,Random forest,Ridge
noise_sigma,,,,
0.00,0.318,0.329,0.311,0.396
0.05,0.369,0.372,0.361,0.430
0.10,0.486,0.475,0.478,0.533
0.20,0.802,0.746,0.803,0.833
0.40,1.529,1.483,1.544,1.625


model,HistGB,MLP,Random forest,Ridge
extra_noise_features,,,,
0,0.318,0.329,0.311,0.396
10,0.319,0.481,0.310,0.396
50,0.324,0.723,0.314,0.400
200,0.320,1.321,0.324,0.397


In [25]:
#7. Comparison with the authors' results and summary

best = B_chrono.drop("Linear (irradiation only)").RMSE.idxmin()
summary = pd.DataFrame([
    ["Paper – baseline LSTM (reported)", "not described", PAPER_BASE["RMSE"], PAPER_BASE["MAE"], PAPER_BASE["R2"], np.nan],
    ["Paper – X-LSTM-EO (reported)", "not described", PAPER["RMSE"], PAPER["MAE"], PAPER["R2"], np.nan],
    ["Floor: best any weather-only model (paper table, in-sample)", "—", floor["RMSE"], floor["MAE"], floor["R2"], np.nan],
    ["Mine: untuned decision tree (Exp A)", "random 80/20", *A.groupby('model')[["RMSE","MAE","R2"]].mean().loc["Decision tree"].values, A.groupby('model').RMSE_day.mean()["Decision tree"]],
    ["Mine: untuned linear, 3 inputs (Exp A)", "random 80/20", *A.groupby('model')[["RMSE","MAE","R2"]].mean().loc["Linear (3 weather inputs)"].values, A.groupby('model').RMSE_day.mean()["Linear (3 weather inputs)"]],
    [f"Mine: {best} (Exp B, tuned on val)", "chronological", *B_chrono.loc[best, ["RMSE", "MAE", "R2", "RMSE_day"]].values],
    ["Mine: oracle floor on test days (Exp B)", "chronological", floor_row["RMSE"], floor_row["MAE"], floor_row["R2"], floor_row["RMSE_day"]],
], columns=["Model", "Split", "RMSE", "MAE", "R2", "RMSE (daytime)"]).round(3)
display(summary); summary.to_csv(f"{OUT_DIR}/summary_vs_paper.csv", index=False)

fig, ax = plt.subplots(figsize=(8.5, 3.8))
lab = ["Paper\nbaseline LSTM", "Paper\nX-LSTM-EO", "Floor\n(weather-only)", "Mine: tree\n(random)", "Mine: linear\n(random)", f"Mine: best\n(chrono)"]
val = summary.RMSE.values[:6]; col = ["#9ca3af", "#6b7280", "#d97706", "#2563eb", "#2563eb", "#16a34a"]
ax.bar(lab, val, color=col); [ax.text(i, v + .03, f"{v:.2f}", ha="center", fontsize=8) for i, v in enumerate(val)]
ax.set_ylabel("RMSE (DC_POWER / 1000)"); ax.set_title("Untuned simple models vs the paper's tuned LSTM"); ax.tick_params(axis="x", labelsize=7.5)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig4_vs_paper.png"); plt.show()

o1 = C_res[1][0]; o4 = C_res[4][0]
print("\nKEY FINDINGS (computed)")
print(f"1. Untuned simple models on the paper's table: best RMSE {best_A:.3f} vs paper tuned LSTM 0.46 (floor {floor['RMSE']:.3f}).")
print(f"2. Chronological split inflates error by {gap.min():.0f}%-{gap.max():.0f}% vs random split on the same data.")
print(f"3. Daytime RMSE is {100*(B_chrono.RMSE_day/B_chrono.RMSE-1).median():.0f}% higher than all-row RMSE (median over models, Exp B) - night zeros flatter averages.")
print(f"4. 15-min-ahead plant forecast skill vs persistence (daytime): " + ", ".join(f"{k}: {v:.2f}" for k, v in o1.filter(like='skill').iloc[:, 0].drop('Persistence').items()))
print(f"5. 1-h-ahead skill vs persistence (daytime): " + ", ".join(f"{k}: {v:.2f}" for k, v in o4.filter(like='skill').iloc[:, 0].drop('Persistence').items()))

,Model,Split,RMSE,MAE,R2,RMSE (daytime)
0,Paper – baseline LSTM (reported),not described,2.200,1.150,0.670,NaN
1,Paper – X-LSTM-EO (reported),not described,0.460,0.229,0.990,NaN
2,Floor: best any weather-only model (paper tabl...,—,0.425,0.159,0.989,NaN
3,Mine: untuned decision tree (Exp A),random 80/20,0.457,0.170,0.987,0.609
4,"Mine: untuned linear, 3 inputs (Exp A)",random 80/20,0.549,0.260,0.982,0.730
5,"Mine: Gradient boosting (HistGB) (Exp B, tuned...",chronological,0.588,0.194,0.977,0.793
6,Mine: oracle floor on test days (Exp B),chronological,0.540,0.153,0.980,0.729



KEY FINDINGS (computed)
1. Untuned simple models on the paper's table: best RMSE 0.457 vs paper tuned LSTM 0.46 (floor 0.425).
2. Chronological split inflates error by 4%-12% vs random split on the same data.
3. Daytime RMSE is 35% higher than all-row RMSE (median over models, Exp B) - night zeros flatter averages.
4. 15-min-ahead plant forecast skill vs persistence (daytime): Climatology: -0.27, Ridge (all features): 0.05, Random forest: 0.09, Gradient boosting (HistGB): 0.09, MLP (32-16): 0.10
5. 1-h-ahead skill vs persistence (daytime): Climatology: 0.28, Ridge (all features): 0.22, Random forest: 0.33, Gradient boosting (HistGB): 0.34, MLP (32-16): 0.34
